# Master Feature Engineering Notebook
All features applied in sequence to avoid dependency issues.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
import nbimporter
import sys
from pathlib import Path
warnings.filterwarnings('ignore')

# All files are in Feature_engineering folder
sys.path.insert(0, str(Path().resolve().parent / 'Feature_engineering'))

print("Path:", Path().resolve().parent / 'Feature_engineering')
print("Exists:", (Path().resolve().parent / 'Feature_engineering').exists())

sys.path.insert(0, str(Path().resolve()))

ModuleNotFoundError: No module named 'nbimporter'

## Import All Feature Functions

In [ ]:

#* import all functions from 4 notebooks

from interaction_machine_features import (
    create_physics_features,
    create_interaction_features,
    create_machine_type_features
)

from outlier_threshold import (
    outlier_analysis,
    threshold_risk_flags
)

from statistical_and_risk_flag_analysis import (
    create_statistical_features,
    create_risk_flag_features
)

from failure_proximity import (
    create_polynomial_features,
    create_failure_proximity_features
)

print("All functions imported successfully!")

All functions imported successfully!


## Load Dataset

In [ ]:
BASE_DIR = Path().resolve().parent
df = pd.read_csv(BASE_DIR / 'Dataset' / 'ai4i2020_cleaned.csv')

#& create copy to prevent overriding
df_feat = df.copy()

print("Shape:", df_feat.shape)
df_feat.head(3)

Shape: (10000, 12)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,0,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,0,298.1,308.5,1498,49.4,5,0,0,0,0,0,0


## 1. Physics Features
> Run first to create dependencies

In [ ]:
df_feat = create_physics_features(df_feat)
print("Columns added:", ['power','temp_diff','wear_rate','torque_per_wear','strain_index','power_per_temp'])

Physics features created!
          power  temp_diff  wear_rate  torque_per_wear  strain_index  \
count  10000.00    10000.0   10000.00         10000.00      10000.00   
mean   59967.15       10.0       0.07             1.32       4314.66   
std    10193.09        1.0       0.04             4.74       2826.57   
min    10966.80        7.6       0.00             0.03          0.00   
25%    53105.40        9.3       0.03             0.24       1963.65   
50%    59883.90        9.8       0.07             0.37       4012.95   
75%    66873.75       11.0       0.11             0.74       6279.00   
max    99980.40       12.1       0.19            68.50      16497.00   

       power_per_temp  
count        10000.00  
mean           193.45  
std             32.90  
min             35.61  
25%            171.25  
50%            193.12  
75%            215.56  
max            324.72  
Columns added: ['power', 'temp_diff', 'wear_rate', 'torque_per_wear', 'strain_index', 'power_per_temp']


## 2. Threshold Risk Flags
 Needs power, temp_diff, strain_index from Physics features. Creates HDF, PWF, OSF, TWF flags needed for proximity features later.

In [ ]:
df_feat = threshold_risk_flags(df_feat)
print("Columns added:", ['hdf_risk_flag','pwf_risk_flag','osf_risk_flag','twf_risk_flag'])

Shape: (10000, 18)
Derived features ready: power, temp_diff, strain_index

=== Threshold Based Flags ===
  hdf_risk_flag : 115
  pwf_risk_flag : 500
  osf_risk_flag : 98
  twf_risk_flag : 801
Columns added: ['hdf_risk_flag', 'pwf_risk_flag', 'osf_risk_flag', 'twf_risk_flag']


## 3. Outlier Analysis Flags
 Adds IQR-based outlier flags for RPM and Torque.

In [ ]:
df_feat = outlier_analysis(df_feat)
print("Columns added:", ['rpm_outlier_flag','torque_outlier_flag'])

=== IQR Method ===
Rotational speed [rpm]
  Outlier rows failure rate : 8.37%
  Normal  rows failure rate : 3.17%

Torque [Nm]
  Outlier rows failure rate : 89.86%
  Normal  rows failure rate : 2.79%

Feature                           IQR
----------------------------------------
Rotational speed [rpm]            418 
Torque [Nm]                        69 


Columns added: ['rpm_outlier_flag', 'torque_outlier_flag']


## 4. Interaction Features
 Needs power, temp_diff from hysics features. Creates combined features

In [ ]:
df_feat = create_interaction_features(df_feat)
print("Columns added:", ['thermal_torque_stress','rotational_load_on_worn_tool','power_absorbed_by_wear','thermal_stress_on_worn_tool','speed_heat_imbalance'])

Interaction features created!
       thermal_torque_stress  rotational_load_on_worn_tool  \
count               10000.00                      10000.00   
mean                12395.96                    6471441.77   
std                  3090.38                    4016716.16   
min                  1170.40                          0.00   
25%                 10288.20                    3096388.80   
50%                 12432.13                    6287880.10   
75%                 14493.98                    9489756.50   
max                 23868.56                   20819214.00   

       power_absorbed_by_wear  thermal_stress_on_worn_tool  \
count                10000.00                     10000.00   
mean               6471441.77                      1079.09   
std                4016716.16                       647.75   
min                      0.00                         0.00   
25%                3096388.80                       524.40   
50%                6287880.10          

## 5. Machine Type Features

In [ ]:
df_feat = create_machine_type_features(df_feat)
print("Columns added:", ['is_low_quality','is_high_quality'])

Machine type features created!
   is_low_quality  is_high_quality
0               0                0
1               1                0
2               1                0
3               1                0
4               1                0
Columns added: ['is_low_quality', 'is_high_quality']


## 6. Statistical Features

In [ ]:
df_feat = create_statistical_features(df_feat)
print("Columns added:", ['rpm_zscore','torque_zscore','power_zscore','wear_zscore','temp_diff_zscore'])

Z-SCORE FEATURES CREATED
  Feature                       Mean      Std   Extreme (>3σ)
  ----------------------------------------------------------
  rpm_zscore                   -0.00     1.00             164
  torque_zscore                 0.00     1.00              25
  power_zscore                 -0.00     1.00              30
  wear_zscore                   0.00     1.00               0
  temp_diff_zscore             -0.00     1.00               0
Columns added: ['rpm_zscore', 'torque_zscore', 'power_zscore', 'wear_zscore', 'temp_diff_zscore']


## 7. General Risk Flags

In [ ]:
df_feat = create_risk_flag_features(df_feat)
print("Columns added:", ['high_wear_flag','high_torque_flag','low_rpm_flag','high_temp_flag','power_anomaly_flag'])

RISK FLAG FEATURES CREATED
  Flag                       Flagged   % Data  Actual Fail  Precision
  ------------------------------------------------------------------
  high_wear_flag                 762     7.6%          118        15%
  high_torque_flag               236     2.4%           99        42%
  low_rpm_flag                  1847    18.5%          236        13%
  high_temp_flag                   0     0.0%            0         0%
  power_anomaly_flag             453     4.5%          130        29%
Columns added: ['high_wear_flag', 'high_torque_flag', 'low_rpm_flag', 'high_temp_flag', 'power_anomaly_flag']


## 8. Polynomial Features

In [ ]:
df_feat = create_polynomial_features(df_feat)
print("Columns added:", ['torque_squared','wear_squared','power_squared','temp_diff_squared','sqrt_power','sqrt_wear','log_power','log_strain'])

Polynomial features created succesfully!
Columns added: ['torque_squared', 'wear_squared', 'power_squared', 'temp_diff_squared', 'sqrt_power', 'sqrt_wear', 'log_power', 'log_strain']


## 9. Failure Proximity Features

In [ ]:
df_feat = create_failure_proximity_features(df_feat)
print("Columns added:", ['distance_to_twf','distance_to_hdf_temp','distance_to_hdf_rpm','distance_to_pwf_lower','distance_to_pwf_upper','distance_to_osf','risk_score'])

Failure proximity features created succesfully!
Columns added: ['distance_to_twf', 'distance_to_hdf_temp', 'distance_to_hdf_rpm', 'distance_to_pwf_lower', 'distance_to_pwf_upper', 'distance_to_osf', 'risk_score']


## Final Summary

In [ ]:
original_cols = df.columns.tolist()
new_cols = [c for c in df_feat.columns if c not in original_cols]

print(f"Original features  : {len(original_cols)}")
print(f"Engineered features: {len(new_cols)}")
print(f"Total              : {len(df_feat.columns)}")
print(f"\nAll engineered features:")
for i, c in enumerate(new_cols, 1):
    print(f"  {i:02d}. {c}")

Original features  : 12
Engineered features: 44
Total              : 56

All engineered features:
  01. power
  02. temp_diff
  03. wear_rate
  04. torque_per_wear
  05. strain_index
  06. power_per_temp
  07. hdf_risk_flag
  08. pwf_risk_flag
  09. osf_risk_flag
  10. twf_risk_flag
  11. rpm_outlier_flag
  12. torque_outlier_flag
  13. thermal_torque_stress
  14. rotational_load_on_worn_tool
  15. power_absorbed_by_wear
  16. thermal_stress_on_worn_tool
  17. speed_heat_imbalance
  18. is_low_quality
  19. is_high_quality
  20. rpm_zscore
  21. torque_zscore
  22. power_zscore
  23. wear_zscore
  24. temp_diff_zscore
  25. high_wear_flag
  26. high_torque_flag
  27. low_rpm_flag
  28. high_temp_flag
  29. power_anomaly_flag
  30. torque_squared
  31. wear_squared
  32. power_squared
  33. temp_diff_squared
  34. sqrt_power
  35. sqrt_wear
  36. log_power
  37. log_strain
  38. distance_to_twf
  39. distance_to_hdf_temp
  40. distance_to_hdf_rpm
  41. distance_to_pwf_lower
  42. distan

In [ ]:

#todo round numeric values to 4 digits
numeric_cols = df_feat.select_dtypes(include='number').columns
df_feat[numeric_cols] = df_feat[numeric_cols].round(4)

In [ ]:
df_feat

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,...,sqrt_wear,log_power,log_strain,distance_to_twf,distance_to_hdf_temp,distance_to_hdf_rpm,distance_to_pwf_lower,distance_to_pwf_upper,distance_to_osf,risk_score
0,1,298.1,308.6,1551,42.8,0,0,0,0,0,...,0.0000,11.1032,0.0000,200,1.9,171,26382.8,13617.2,12000.0,0
1,0,298.2,308.7,1408,46.3,3,0,0,0,0,...,1.7321,11.0851,4.9409,197,1.9,28,25190.4,14809.6,10861.1,0
2,0,298.1,308.5,1498,49.4,5,0,0,0,0,...,2.2361,11.2119,5.5134,195,1.8,118,34001.2,5998.8,10753.0,0
3,0,298.2,308.6,1433,39.5,7,0,0,0,0,...,2.6458,10.9438,5.6258,193,1.8,53,16603.5,23396.5,10723.5,0
4,0,298.2,308.7,1408,40.0,9,0,0,0,0,...,3.0000,10.9388,5.8889,191,1.9,28,16320.0,23680.0,10640.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1,298.8,308.4,1604,29.5,14,0,0,0,0,...,3.7417,10.7647,6.0259,186,1.0,224,7318.0,32682.0,11587.0,0
9996,2,298.9,308.4,1632,31.8,17,0,0,0,0,...,4.1231,10.8570,6.2945,183,0.9,252,11897.6,28102.4,12459.4,0
9997,1,299.0,308.6,1645,33.4,22,0,0,0,0,...,4.6904,10.9141,6.6010,178,1.0,265,14943.0,25057.0,11265.2,0
9998,2,299.0,308.7,1408,48.5,25,0,0,0,0,...,5.0000,11.1315,7.1013,175,1.1,28,28288.0,11712.0,11787.5,0


## 13. Export

In [ ]:
df_feat.to_csv(BASE_DIR / 'Dataset' / 'ai4i2020_features.csv', index=False)
print("Saved successfully!")
print("Shape:", df_feat.shape)

Saved successfully!
Shape: (10000, 56)
